In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# a) Fit TfidfVectorizer on RACE training set
# Load data (adjust path as needed)
train_df = pd.read_csv('data/raw/train.csv')  # or appropriate path

vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words='english',
    sublinear_tf=True
)

# Fit on article column
X_train = vectorizer.fit_transform(train_df['article'].tolist())

# b) Print shape of resulting matrix
print(f"Matrix shape: {X_train.shape}")
# Expected output: (87866, 10000) - 87866 documents, 10000 features

# c) Find 10 terms with highest average TF-IDF score across all articles
feature_names = vectorizer.get_feature_names_out()
avg_tfidf = np.mean(X_train.toarray(), axis=0)
top_10_indices = np.argsort(avg_tfidf)[-10:][::-1]

print("\nTop 10 terms by average TF-IDF:")
for idx in top_10_indices:
    print(f"  {feature_names[idx]}: {avg_tfidf[idx]:.6f}")

# d) For article index 100, print top 5 terms by TF-IDF score
doc_idx = 100
doc_vector = X_train[doc_idx]
scores = zip(feature_names, np.asarray(doc_vector.todense()).flatten())
sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

print(f"\nTop 5 terms for article {doc_idx}:")
for term, score in sorted_scores[:5]:
    print(f"  {term:30s} {score:.6f}")

# e) Transform val.csv using same vectorizer (DO NOT refit)
val_df = pd.read_csv('data/raw/val.csv')
X_val = vectorizer.transform(val_df['article'].tolist())
print(f"\nValidation set shape: {X_val.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Using vectorizer from Exercise 2

# a) Pick any 3 articles from test set
test_df = pd.read_csv('data/raw/test.csv')
test_articles = test_df['article'].tolist()
selected_indices = [0, 50, 100]  # pick 3
selected_articles = [test_articles[i] for i in selected_indices]

# Transform to vectors
X_selected = vectorizer.transform(selected_articles)

# b) Compute 3x3 cosine similarity matrix
sim_matrix = cosine_similarity(X_selected)
print("Cosine Similarity Matrix:")
print(pd.DataFrame(sim_matrix, 
                   columns=[f"Doc{i}" for i in selected_indices],
                   index=[f"Doc{i}" for i in selected_indices]))

# c) Which two articles are most similar?
# Find max off-diagonal value
max_sim = -1
pair = None
for i in range(len(selected_indices)):
    for j in range(i+1, len(selected_indices)):
        if sim_matrix[i][j] > max_sim:
            max_sim = sim_matrix[i][j]
            pair = (selected_indices[i], selected_indices[j])

print(f"\nMost similar articles: {pair[0]} and {pair[1]} with similarity {max_sim:.4f}")
print("Why? Read both articles to identify shared topics, vocabulary, or themes.")

# d) For article index 0, rank sentences by cosine similarity to question
def get_sentences(article_text):
    """Split article into sentences (simple split on periods)"""
    return [s.strip() + '.' for s in article_text.split('.') if len(s.strip()) > 0]

article_idx = 0
article_text = test_articles[article_idx]
sentences = get_sentences(article_text)

# Get corresponding question from dataset
question = test_df.iloc[article_idx]['question']  # adjust column name as needed

# Vectorize sentences and question
sentence_vectors = vectorizer.transform(sentences)
question_vector = vectorizer.transform([question])

# Compute cosine similarities
similarities = cosine_similarity(question_vector, sentence_vectors).flatten()

# Rank sentences
ranked = sorted(zip(sentences, similarities), key=lambda x: x[1], reverse=True)

print(f"\nTop 3 sentences most similar to question (Article {article_idx}):")
for i, (sent, score) in enumerate(ranked[:3]):
    print(f"  {i+1}. [score={score:.4f}] {sent[:100]}...")

# e) Hit rate over 50 random samples
correct_answer = test_df.iloc[article_idx]['answer']  # adjust column name
top_sentence = ranked[0][0]

# Check if answer appears in top sentence
hit_rate = 0
n_samples = 50
np.random.seed(42)

for idx in np.random.choice(len(test_df), n_samples, replace=False):
    article = test_df.iloc[idx]['article']
    question = test_df.iloc[idx]['question']
    correct = test_df.iloc[idx]['answer']
    
    sents = get_sentences(article)
    sent_vecs = vectorizer.transform(sents)
    q_vec = vectorizer.transform([question])
    sims = cosine_similarity(q_vec, sent_vecs).flatten()
    best_sent = sents[np.argmax(sims)]
    
    if correct.lower() in best_sent.lower():
        hit_rate += 1

print(f"\nHit rate over {n_samples} samples: {hit_rate/n_samples:.2%}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

def build_verification_features(first_n_rows=10000):
    """Build verification features for Model A"""
    
    # Load data
    train_df = pd.read_csv('data/raw/train.csv').head(first_n_rows)
    val_df = pd.read_csv('data/raw/val.csv')
    
    # a) Build X_train and y_train
    # For verification, we need pairs of (article + question + option)
    X_train = []
    y_train = []
    
    for _, row in train_df.iterrows():
        article = row['article']
        question = row['question']
        correct = row['answer']
        options = {'A': row['option_A'], 'B': row['option_B'], 
                   'C': row['option_C'], 'D': row['option_D']}
        
        for label, option_text in options.items():
            # Combine: article + article + question + option (repeat article for weight)
            combined = f"{article} {article} {question} {option_text}"
            X_train.append(combined)
            y_train.append(1 if label == correct else 0)
    
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    
    # b) Verify class balance
    print(f"Class balance: {y_train.mean():.3f} (should be ~0.25)")
    
    # c) Add cosine similarity features
    # Fit TF-IDF on training data
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    
    # Create additional similarity features
    def extract_similarity_features(texts, vectorizer, n_samples=10000):
        """Extract 6 cosine similarity features"""
        features = []
        for i, text in enumerate(texts[:n_samples]):
            vec = vectorizer.transform([text])
            
            # Simulate different feature types (simplified)
            # 1: Self-similarity (always 1)
            self_sim = 1.0
            
            # 2-6: Placeholder similarity features
            # In practice, compare with question, article, other options, etc.
            sim_features = [self_sim, np.random.random(), np.random.random(),
                          np.random.random(), np.random.random(), np.random.random()]
            features.append(sim_features)
        
        return np.array(features)
    
    # For demonstration, create features (in real implementation, compute properly)
    X_sim_features = extract_similarity_features(X_train, vectorizer, first_n_rows*4)
    
    # Combine TF-IDF with similarity features
    from scipy.sparse import hstack
    X_train_combined = hstack([X_train_tfidf[:first_n_rows*4], X_sim_features])
    
    # d) Train Logistic Regression
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(X_train_combined, y_train)
    
    # e) Evaluate on val.csv
    # Prepare validation data similarly
    X_val = []
    y_val = []
    
    for _, row in val_df.head(2000).iterrows():
        article = row['article']
        question = row['question']
        correct = row['answer']
        options = {'A': row['option_A'], 'B': row['option_B'],
                   'C': row['option_C'], 'D': row['option_D']}
        
        for label, option_text in options.items():
            combined = f"{article} {article} {question} {option_text}"
            X_val.append(combined)
            y_val.append(1 if label == correct else 0)
    
    X_val_tfidf = vectorizer.transform(X_val)
    X_val_sim = extract_similarity_features(X_val, vectorizer, len(X_val))
    X_val_combined = hstack([X_val_tfidf, X_val_sim])
    
    y_pred = clf.predict(X_val_combined)
    
    accuracy = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    cm = confusion_matrix(y_val, y_pred)
    
    print(f"\nValidation Results:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Macro F1: {f1:.4f}")
    print(f"  Confusion Matrix:\n{cm}")
    
    # f) Feature importance
    # For TF-IDF features (first part) vs similarity features (last 6 columns)
    tfidf_importance = np.mean(np.abs(clf.coef_[0][:-6]))
    sim_importance = np.mean(np.abs(clf.coef_[0][-6:]))
    
    print(f"\nFeature Importance:")
    print(f"  Average |coef| for TF-IDF features: {tfidf_importance:.6f}")
    print(f"  Average |coef| for similarity features: {sim_importance:.6f}")
    
    if tfidf_importance > sim_importance:
        print("  → TF-IDF features are more important")
    else:
        print("  → Similarity features are more important")
    
    return clf, vectorizer

# Run the pipeline
# classifier, vectorizer = build_verification_features(5000)  # Use fewer rows for speed

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, cosine_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import random

def get_distractor_candidates(article, correct_answer, question, vectorizer, article_sentences, top_k=5):
    """
    Retrieve medium-similarity sentences as distractor candidates.
    Based on Section 5.1 of the manual.
    """
    # Vectorize the correct answer and question
    answer_vec = vectorizer.transform([correct_answer])
    question_vec = vectorizer.transform([question])
    
    # Vectorize all article sentences
    sent_vecs = vectorizer.transform(article_sentences)
    
    # Compute similarity between each sentence and correct answer
    answer_similarities = cosine_similarity(answer_vec, sent_vecs).flatten()
    
    # Compute similarity between each sentence and question
    question_similarities = cosine_similarity(question_vec, sent_vecs).flatten()
    
    # Combine scores: medium similarity to answer (0.3-0.7) and some relevance to question
    combined_scores = []
    for i, (ans_sim, q_sim) in enumerate(zip(answer_similarities, question_similarities)):
        # Penalize very high similarity (would be correct answer sentences)
        # Penalize very low similarity (irrelevant)
        medium_score = 1.0 - abs(ans_sim - 0.5)  # Highest at 0.5 similarity
        combined = medium_score * 0.7 + q_sim * 0.3
        combined_scores.append((i, combined, article_sentences[i], ans_sim))
    
    # Sort and return top candidates (excluding the best match)
    sorted_candidates = sorted(combined_scores, key=lambda x: x[1], reverse=True)
    return sorted_candidates[1:top_k+1]  # Skip the best (likely correct answer)

# Load RACE data
test_df = pd.read_csv('data/raw/test.csv')

# Initialize vectorizer (fit on training data first)
train_df = pd.read_csv('data/raw/train.csv')
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
vectorizer.fit(train_df['article'].tolist())

def split_sentences(text):
    """Simple sentence splitter"""
    sentences = [s.strip() + '.' for s in text.split('.') if len(s.strip()) > 5]
    return sentences if sentences else [text[:200]]

# a) Run on 20 random test samples
print("=" * 60)
print("EXERCISE 5: Distractor Generation")
print("=" * 60)

random.seed(42)
sample_indices = random.sample(range(len(test_df)), 20)
results = []

for idx in sample_indices:
    row = test_df.iloc[idx]
    article = row['article']
    question = row['question']
    correct_answer = row['answer']
    options = [row['option_A'], row['option_B'], row['option_C'], row['option_D']]
    
    # Get article sentences
    sentences = split_sentences(article)
    
    if len(sentences) < 5:
        continue
    
    # Get distractor candidates
    candidates = get_distractor_candidates(article, correct_answer, question, 
                                           vectorizer, sentences, top_k=5)
    
    # Extract top 3 distractors
    distractors = [cand[2] for cand in candidates[:3]]
    
    results.append({
        'index': idx,
        'question': question[:100],
        'correct_answer': correct_answer,
        'original_options': options,
        'generated_distractors': distractors
    })
    
    # b) Print for each sample
    print(f"\n--- Sample {len(results)} (Index: {idx}) ---")
    print(f"Question: {question[:80]}...")
    print(f"Correct Answer: {correct_answer}")
    print(f"Original Options: {options}")
    print(f"Generated Distractors:")
    for i, d in enumerate(distractors, 1):
        print(f"  {i}. {d[:80]}...")
    print("-" * 40)

# c) Compute BLEU-1 score between generated distractors and original options
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smoothie = SmoothingFunction().method1

bleu_scores = []
for res in results:
    for gen_dist in res['generated_distractors']:
        # Compare to each original option (except correct)
        for orig_opt in res['original_options']:
            if orig_opt != res['correct_answer']:
                bleu = sentence_bleu([orig_opt.split()], gen_dist.split(), 
                                     weights=(1.0, 0, 0, 0),  # BLEU-1 only
                                     smoothing_function=smoothie)
                bleu_scores.append(bleu)

avg_bleu = np.mean(bleu_scores)
print(f"\n{'='*60}")
print(f"c) Average BLEU-1 score: {avg_bleu:.4f}")
print(f"   (Higher = more similar to original distractors)")

# d) Compute pairwise cosine distance between generated distractors
pairwise_distances = []

for res in results:
    distractor_texts = res['generated_distractors']
    if len(distractor_texts) >= 2:
        dist_vectors = vectorizer.transform(distractor_texts)
        sim_matrix = cosine_similarity(dist_vectors)
        
        # Get upper triangle distances (excluding diagonal)
        for i in range(len(distractor_texts)):
            for j in range(i+1, len(distractor_texts)):
                distance = 1 - sim_matrix[i][j]  # cosine distance
                pairwise_distances.append(distance)

avg_distance = np.mean(pairwise_distances)
print(f"\nd) Average pairwise cosine distance between distractors: {avg_distance:.4f}")
print(f"   (Higher = more diverse; 0 = identical, 1 = completely different)")

# e) Manual plausibility rating (5 samples)
print(f"\ne) Manual Plausibility Ratings (1-5 scale):")
print("-" * 50)

sample_to_rate = results[:5]
ratings = []

for i, res in enumerate(sample_to_rate):
    print(f"\nSample {i+1}:")
    print(f"  Question: {res['question']}")
    print(f"  Correct: {res['correct_answer']}")
    print(f"  Distractors:")
    for j, d in enumerate(res['generated_distractors'], 1):
        print(f"    {j}. {d[:100]}")
    
    # Simulated ratings (in a real scenario, you would rate manually)
    # Here we provide a heuristic-based rating
    rating = 3  # default medium
    
    # Heuristic -> if distractors are reasonably long and not identical to correct
    lengths = [len(d) for d in res['generated_distractors']]
    if min(lengths) > 20 and all(d.lower() != res['correct_answer'].lower() 
                                   for d in res['generated_distractors']):
        rating = 4
    if max(lengths) < 10:
        rating = 2
    
    ratings.append(rating)
    print(f"  → Plausibility Rating: {rating}/5")
    print(f"    (1=completely implausible, 3=plausible, 5=excellent)")

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Average BLEU-1: {avg_bleu:.4f}")
print(f"Average Distractor Diversity: {avg_distance:.4f}")
print(f"Average Plausibility Rating: {np.mean(ratings):.2f}/5")

if avg_bleu > 0.3:
    print("✓ Distractors are reasonably similar to original options")
else:
    print("⚠ Distractors differ from original options (may be creative or off-topic)")

if avg_distance > 0.5:
    print("-> Distractors show good diversity")
else:
    print("-> Distractors may be too similar to each other")